# This notebook aims to test the different components

In [ ]:
from pprint import pprint

## Test the MCP Server

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "teradata": {
        "transport": "http",
        "url": "https://dmproject.myddns.me/mcpce/",
    }
})

tools = await client.get_tools()
print([getattr(t, "name", str(t)) for t in tools])

## Test the LLM API

In [ ]:
from dotenv import load_dotenv
# Load API key safely from .env
load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
import os

if True:
    MODEL = ChatOpenAI(
        model="mistralai/Ministral-3-14B-Instruct-2512",
        base_url="https://api-dmproject.myddns.me/v1",
        api_key=os.environ["LOCAL_TOKEN"],  # or your mapped env var like FS_API_KEY, LOCAL_TOKEN
        temperature=0,
    )
else:
    MODEL = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
MODEL.invoke("hello")

## Test Agent

In [ ]:
from agent_builder import create_sql_agent_with_mcp
from pathlib import Path

MCP_SERVERS = {
    "teradata": {
        "transport": "http",  
        "url": "https://dmproject.myddns.me/mcpce/",
        
        # Uncomment ONLY if your MCP server requires auth
        # "headers": {
        #     "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        # }
    }
}

agent = await create_sql_agent_with_mcp(
    model         = MODEL,
    skills_root   = Path("./skills"),
    mcp_servers   = MCP_SERVERS,
    system_prompt = """
You are a SQL assistant.

Rules:
- When a request matches a skill, load the skill first.
- Then use MCP tools to interact with the database.
- Do NOT invent schema.
- Be concise and correct.
""",
)

In [ ]:
import uuid
from pprint import pprint
from print_agent_output import print_agent_timeline, to_dataframe

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

result = await agent.ainvoke(
    {
        "messages": [
            {"role": "user", "content": "What tables are in the database DB_SOURCE?"}
        ]
    },
    config,
)

print_agent_timeline(result)
to_dataframe(result)

## Test answer

In [ ]:
from agent_ui import build_skill_answer

In [ ]:
skill_answer = build_skill_answer(agent=agent, return_trace=True)

In [ ]:
answer, trace = await skill_answer("Segment customers by spending behavior.")

print("ANSWER:\n", answer)
print("\nTRACE:\n", trace)